# Phase 10: Graph Builder Debug

Validates the graph builder pipeline:
- Build a `GraphResponse` from a real claim end-to-end
- Inspect nodes: types, colors, sizes, URLs, confidence
- Inspect edges: types, weights, widths, dashed styles
- Run all 5 Phase 10 checklist verifications
- Save graph JSON and print structure summary

Uses `VerifyClaimService.build_default()` + `GraphBuilderService` — no manual wiring needed.

In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()

from backend.app.services.verify_claim_service import VerifyClaimService
from backend.app.services.graph_builder_service import GraphBuilderService
from backend.app.utils.constants import (
    NODE_COLOR_MAIN_VERIFIED, NODE_COLOR_MAIN_REJECTED, NODE_COLOR_MAIN_NEI,
    NODE_SIZE_MAIN_CLAIM, NODE_SIZE_DIRECT_EVIDENCE, NODE_SIZE_WEAK_EVIDENCE,
    EDGE_COLOR_SUPPORTS, EDGE_COLOR_REFUTES, EDGE_COLOR_CORRELATED, EDGE_COLOR_INSUFFICIENT,
)

print('Imports OK')

/Users/vraj21/Desktop/Projects/Fact Checker/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Imports OK


## Load Services

In [2]:
verify_svc = VerifyClaimService.build_default()
builder    = GraphBuilderService()
print('Services ready.')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1485.41it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[WikipediaRetriever] FAISS loaded: 40,574 vectors


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 2479.75it/s, Materializing param=pooler.dense.weight]                                     
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Groq] Model: llama-3.1-8b-instant
Services ready.


## Build Graph — Single Claim

In [3]:
CLAIM = "The Eiffel Tower is located in Paris, France."

print(f'Claim : {CLAIM}')
result = verify_svc.verify(CLAIM, use_cache=False)
graph  = builder.build(result)

print(f'  Sources retrieved : {len(result.sources)}')
print(f'  LLM verdict       : {result.llm_result.overall_verdict} (conf={result.llm_result.confidence:.2f})')
print(f'  Final verdict     : {result.confidence_output.overall_verdict} ({result.confidence_output.overall_confidence:.2f})')
print(f'  Graph nodes       : {graph.metadata.total_nodes}')
print(f'  Graph edges       : {graph.metadata.total_edges}')

Claim : The Eiffel Tower is located in Paris, France.
[Groq] Generating... (model=llama-3.1-8b-instant, max_tokens=512)
[Groq] Generation complete.
  Sources retrieved : 6
  LLM verdict       : supported (conf=1.00)
  Final verdict     : verified (0.74)
  Graph nodes       : 6
  Graph edges       : 5


## Inspect Nodes

In [5]:
print(f"{'ID':<18} {'TYPE':<24} {'VERDICT':<14} {'CONF':<6} {'SIZE':<6} {'COLOR':<10} {'URL':<5} MAIN")
print('-' * 100)
for node in graph.nodes:
    main_marker = '<-- MAIN' if node.is_main_claim else ''
    url_ok      = 'YES' if node.best_source_url else 'MISSING'
    print(f"{node.node_id:<18} {node.node_type:<24} {node.verdict:<14} "
          f"{node.confidence:<6.2f} {node.size:<6.0f} {node.color:<10} {url_ok:<5} {main_marker}")

print(f'\nNode counts: support={graph.metadata.support_node_count} '
      f'refute={graph.metadata.refute_node_count} '
      f'context={graph.metadata.context_node_count} '
      f'factcheck={graph.metadata.factcheck_node_count} '
      f'insufficient={graph.metadata.insufficient_node_count}')

ID                 TYPE                     VERDICT        CONF   SIZE   COLOR      URL   MAIN
----------------------------------------------------------------------------------------------------
node_main          main_claim               verified       0.74   24     #2E7D32    YES   <-- MAIN
node_ev_01         context_signal           refutes        0.84   7      #4A148C    YES   
node_ev_02         direct_support           supports       0.90   10     #1565C0    YES   
node_ev_03         direct_support           supports       0.90   10     #1565C0    YES   
node_ev_04         insufficient_evidence    refutes        0.68   7      #424242    YES   
node_ev_05         direct_support           refutes        0.80   10     #1565C0    YES   

Node counts: support=3 refute=0 context=1 factcheck=0 insufficient=1


## Inspect Edges

In [6]:
print(f"{'FROM':<12} {'TO':<18} {'TYPE':<14} {'WEIGHT':<8} {'WIDTH':<7} {'DASHED':<8} COLOR")
print('-' * 85)
for edge in graph.edges:
    print(f"{edge.source:<12} {edge.target:<18} {edge.edge_type:<14} "
          f"{edge.weight:<8.2f} {edge.width:<7.1f} {str(edge.dashed):<8} {edge.color}")

from collections import Counter
type_counts = Counter(e.edge_type for e in graph.edges)
print(f'\nEdge type breakdown: {dict(type_counts)}')

FROM         TO                 TYPE           WEIGHT   WIDTH   DASHED   COLOR
-------------------------------------------------------------------------------------
node_main    node_ev_01         correlated     0.84     5.1     True     #8E24AA
node_main    node_ev_02         supports       0.90     5.4     False    #1E88E5
node_main    node_ev_03         supports       0.90     5.4     False    #1E88E5
node_main    node_ev_04         insufficient   0.68     4.2     True     #BDBDBD
node_main    node_ev_05         supports       0.80     4.9     False    #1E88E5

Edge type breakdown: {'correlated': 1, 'supports': 3, 'insufficient': 1}


## Hover Data — What the Frontend Sees

Simulates what `NodeTooltip.jsx` will display when hovering a node.

In [7]:
main_node = next(n for n in graph.nodes if n.is_main_claim)

print('=== MAIN NODE HOVER ===')
print(f'  Verdict     : {main_node.verdict}')
print(f'  Confidence  : {main_node.confidence:.2f}')
print(f'  Explanation : {main_node.short_explanation}')
print(f'  Click URL   : {main_node.best_source_url}')
print(f'  Sources ({len(main_node.top_sources)}):')
for s in main_node.top_sources:
    print(f'    [{s.source_type:<12}] {s.title[:55]}')
    print(f'               trust={s.trust_score:.2f}  relevance={s.relevance_score:.2f}  stance={s.stance_hint}')

print()
print('=== EVIDENCE NODE HOVER (first 3) ===')
for node in [n for n in graph.nodes if not n.is_main_claim][:3]:
    src = node.top_sources[0] if node.top_sources else None
    print(f'  [{node.node_id}] {node.node_type}  verdict={node.verdict}  conf={node.confidence:.2f}')
    if src:
        print(f'    Snippet  : {(src.snippet or "")[:90]}')
        print(f'    Click URL: {node.best_source_url}')
    print(f'    Rationale: {node.short_explanation[:90]}')
    print()

=== MAIN NODE HOVER ===
  Verdict     : verified
  Confidence  : 0.74
  Explanation : Multiple sources confirm the Eiffel Tower's location in Paris, France.
  Click URL   : https://en.wikipedia.org/wiki/Eiffel_Tower
  Sources (5):
    [livewiki    ] Wikipedia: Champ de Mars
               trust=0.80  relevance=1.00  stance=refutes
    [livewiki    ] Wikipedia: Eiffel Tower
               trust=0.80  relevance=0.80  stance=supports
    [livewiki    ] Wikipedia: List of tallest buildings and structures in 
               trust=0.80  relevance=0.80  stance=supports
    [wikipedia   ] Wikipedia: Tower_of_London (sentence 0)
               trust=0.75  relevance=0.50  stance=refutes
    [wikipedia   ] Wikipedia: Place_de_la_Concorde (sentence 0)
               trust=0.75  relevance=0.46  stance=refutes

=== EVIDENCE NODE HOVER (first 3) ===
  [node_ev_01] context_signal  verdict=refutes  conf=0.84
    Snippet  : The Champ de Mars is a large public greenspace in Paris, France, located in the 

## Multi-Claim Test (3 Claims)

In [9]:
TEST_CLAIMS = [
    ("Barack Obama was the 44th president of the United States.", "verified"),
    ("Humans only use 10 percent of their brains.",               "rejected"),
    ("A secret underground city exists beneath Denver Airport.",  "not_enough_info"),
]

print(f"{'#':<3} {'VERDICT':<14} {'CONF':<6} {'NODES':<6} {'EDGES':<6} {'EXPECTED':<14} PASS  CLAIM")
print('-' * 95)
for i, (claim, expected) in enumerate(TEST_CLAIMS, 1):
    res = verify_svc.verify(claim, use_cache=False)
    g   = builder.build(res)
    passed = g.metadata.overall_verdict == expected
    mark   = 'PASS' if passed else 'FAIL'
    print(f"{i:<3} {g.metadata.overall_verdict:<14} {g.metadata.overall_confidence:<6.2f} "
          f"{g.metadata.total_nodes:<6} {g.metadata.total_edges:<6} {expected:<14} {mark:<5} {claim[:45]}")

#   VERDICT        CONF   NODES  EDGES  EXPECTED       PASS  CLAIM
-----------------------------------------------------------------------------------------------
[Groq] Generating... (model=llama-3.1-8b-instant, max_tokens=512)
[Groq] Generation complete.
1   verified       0.78   6      5      verified       PASS  Barack Obama was the 44th president of the Un
[Groq] Generating... (model=llama-3.1-8b-instant, max_tokens=512)
[Groq] Generation complete.
2   not_enough_info 0.28   2      1      rejected       FAIL  Humans only use 10 percent of their brains.
[Groq] Generating... (model=llama-3.1-8b-instant, max_tokens=512)
[Groq] Generation complete.
3   not_enough_info 0.06   3      2      not_enough_info PASS  A secret underground city exists beneath Denv


## Phase 10 Checklist Verification

In [10]:
g = graph
issues = []

# 1. Node count
if g.metadata.total_nodes < 2:
    issues.append(f'Node count too low: {g.metadata.total_nodes}')
else:
    print(f'[PASS] 1. Node count: {g.metadata.total_nodes}')

# 2. Edge count = evidence node count
evidence_count = g.metadata.total_nodes - 1
if g.metadata.total_edges != evidence_count:
    issues.append(f'Edge count {g.metadata.total_edges} != evidence nodes {evidence_count}')
else:
    print(f'[PASS] 2. Edge count: {g.metadata.total_edges} (1 per evidence node)')

# 3. Node fields complete
field_issues = []
for node in g.nodes:
    if not node.node_id: field_issues.append(f'{node.node_id}: missing node_id')
    if not node.color:   field_issues.append(f'{node.node_id}: missing color')
    if not node.verdict: field_issues.append(f'{node.node_id}: missing verdict')
    if not (0 <= node.confidence <= 1): field_issues.append(f'{node.node_id}: confidence OOB')
    if node.short_explanation is None:  field_issues.append(f'{node.node_id}: missing explanation')
if field_issues:
    issues += field_issues
else:
    print(f'[PASS] 3. Node fields complete (id, color, verdict, confidence, explanation)')

# 4. No missing URLs
missing = [n.node_id for n in g.nodes if not n.best_source_url]
if missing:
    issues.append(f'Nodes missing URL: {missing}')
else:
    print(f'[PASS] 4. No missing URLs — all {g.metadata.total_nodes} nodes have best_source_url')

# 5. Node type matches edge type
type_to_edge = {'direct_support':'supports','direct_refute':'refutes',
                'factcheck_review':'refutes','context_signal':'correlated',
                'insufficient_evidence':'insufficient'}
node_map = {n.node_id: n.node_type for n in g.nodes}
type_issues = []
for edge in g.edges:
    ttype = node_map.get(edge.target, '')
    exp   = type_to_edge.get(ttype)
    if edge.edge_type != exp:
        type_issues.append(f'{edge.target}: node_type={ttype} edge_type={edge.edge_type} (expected {exp})')
if type_issues:
    issues += type_issues
else:
    print(f'[PASS] 5. Node types match edge types')

print()
if issues:
    print('ISSUES:')
    for iss in issues: print(f'  [FAIL] {iss}')
else:
    print('All 5 Phase 10 checks passed.')

[PASS] 1. Node count: 6
[PASS] 2. Edge count: 5 (1 per evidence node)
[PASS] 3. Node fields complete (id, color, verdict, confidence, explanation)
[PASS] 4. No missing URLs — all 6 nodes have best_source_url
[PASS] 5. Node types match edge types

All 5 Phase 10 checks passed.


## Save Graph JSON

In [11]:
out_dir  = PROJECT_ROOT / 'data' / 'artifacts' / 'graph_samples'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'graph_eiffel_tower.json'
out_path.write_text(graph.model_dump_json(indent=2))
print(f'Saved: {out_path}')

data = json.loads(graph.model_dump_json())
print(f'\nMetadata:')
print(json.dumps(data['metadata'], indent=2))

main = next(n for n in data['nodes'] if n['is_main_claim'])
preview = {k: v for k, v in main.items() if k != 'top_sources'}
print(f'\nMain node (without top_sources):')
print(json.dumps(preview, indent=2))
print(f'  top_sources: {len(main["top_sources"])} attached')

Saved: /Users/vraj21/Desktop/Projects/Fact Checker/data/artifacts/graph_samples/graph_eiffel_tower.json

Metadata:
{
  "claim_text": "The Eiffel Tower is located in Paris, France.",
  "overall_verdict": "verified",
  "overall_confidence": 0.74,
  "total_nodes": 6,
  "total_edges": 5,
  "support_node_count": 3,
  "refute_node_count": 0,
  "insufficient_node_count": 1,
  "context_node_count": 1,
  "factcheck_node_count": 0,
  "top_support_score": 0.897,
  "top_refute_score": null,
  "retrieval_notes": "6 sources retrieved from: livewiki, wikipedia."
}

Main node (without top_sources):
{
  "node_id": "node_main",
  "node_type": "main_claim",
  "text": "The Eiffel Tower is located in Paris, France.",
  "verdict": "verified",
  "confidence": 0.74,
  "size": 24.0,
  "color": "#2E7D32",
  "best_source_url": "https://en.wikipedia.org/wiki/Eiffel_Tower",
  "short_explanation": "Multiple sources confirm the Eiffel Tower's location in Paris, France.",
  "source_count": 5,
  "is_main_claim": true


## Final Checklist

- [ ] `GraphBuilderService.build()` returns valid `GraphResponse`
- [ ] Main node: `is_main_claim=True`, `node_type=main_claim`, correct color per verdict
- [ ] Verified → green (`#2E7D32`), rejected → red (`#C62828`), NEI → grey (`#757575`)
- [ ] Evidence colors: `direct_support` blue, `direct_refute` orange-red, `factcheck_review` purple
- [ ] `factcheck` source_type always becomes `factcheck_review` node regardless of LLM class
- [ ] Correlated/insufficient edges are dashed; supports/refutes are solid
- [ ] Edge width scales with confidence (thicker = stronger evidence)
- [ ] All nodes have `best_source_url` (click-to-redirect in frontend)
- [ ] Main node `top_sources` ≤ 5 (hover panel in frontend)
- [ ] `short_explanation` on all nodes (tooltip text)
- [ ] `model_dump_json()` valid — ready for FastAPI `GraphResponse`
- [ ] `--mode graph` CLI saves JSON to `data/artifacts/graph_samples/`
- [ ] `pytest tests/unit/test_graph_builder.py` → 57/57 passed